# Section 2: The Agent Loop

*Duration: 20 minutes*

---

Section 1 established that a passive RAG pipeline has no recovery path when retrieval fails. It retrieves, it answers, it moves on. The architecture has no place to inspect, evaluate, or choose a different strategy.

This section builds the control structure that fixes that: the **agent loop**.

The idea is simple. Instead of executing a fixed sequence (retrieve then answer), the model gets a decision to make at each step. It can retrieve, evaluate what came back, rewrite the query and try again, or decide that it cannot answer. The retriever does not go away. It becomes one tool among several, and the model decides when and how to use it.

By the end of this section, the same model and the same retriever will correctly answer questions that the passive pipeline got wrong. Nothing about the model or the data changes. Only the control structure changes.

## 2.1 Setup and Recap

We need the same components from Section 1: the evaluation results, the MaaS endpoint, and the vector store. The cell below loads everything.

In [ ]:
import json
import os
from openai import OpenAI

# Load evaluation results from Section 1
EVAL_PATH = "../prebuilt/eval_results.json"

with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

results = eval_data["results"]
failures = [r for r in results if r["classification"] != "pass"]

print(f"Loaded {len(results)} questions, {len(failures)} failures")
print(f"\nFailing questions:")
for f_ in failures:
    print(f"  {f_['id']}: {f_['question']}")

In [ ]:
# Connect to the MaaS endpoint
api_key  = os.environ.get("MAAS_API_KEY")
base_url = os.environ.get("MAAS_BASE_URL")
model_id = os.environ.get("MAAS_MODEL_ID", "granite-3-2-8b-instruct")

client = OpenAI(api_key=api_key, base_url=base_url)

print(f"Connected to : {base_url}")
print(f"Model        : {model_id}")

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../prebuilt/chroma_db")
collection = chroma_client.get_collection("basic_fantasy_corpus")

print(f"Collection    : {collection.name}")
print(f"Documents     : {collection.count()}")

## 2.2 The Retrieval Evaluator

The first thing the passive pipeline lacks is the ability to judge its own retrieval. It pulls chunks and feeds them to the model without asking: *are these chunks actually useful for this question?*

The fix is a **retrieval evaluator**: a prompt that takes the question and the retrieved chunks and returns a judgment. Not an answer. Just a judgment: are these chunks sufficient?

This is the simplest possible agent capability. The model already knows how to do this. The passive pipeline just never asked.

In [ ]:
EVALUATOR_PROMPT = """You are a retrieval quality evaluator for a rules-reference system.

You will receive a QUESTION and a set of RETRIEVED CHUNKS from a document corpus.

Your job is to judge whether the retrieved chunks contain enough information to
correctly and completely answer the question.

Respond with EXACTLY one of these JSON objects (no other text):

If the chunks contain the answer:
{"verdict": "sufficient", "reason": "<one sentence>"}

If the chunks are related but incomplete:
{"verdict": "partial", "reason": "<one sentence>", "missing": "<what is missing>"}

If the chunks are not relevant to the question:
{"verdict": "irrelevant", "reason": "<one sentence>"}
"""


def evaluate_retrieval(question, chunks, model_client, model_id):
    """Ask the model to judge whether retrieved chunks can answer the question."""
    chunk_text = "\n\n---\n\n".join(chunks)

    response = model_client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": EVALUATOR_PROMPT},
            {"role": "user", "content": f"QUESTION:\n{question}\n\nRETRIEVED CHUNKS:\n{chunk_text}"}
        ],
        temperature=0.0
    )

    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"verdict": "error", "reason": "Could not parse evaluator response", "raw": raw}


print("Evaluator defined.")

Test the evaluator on one passing and one failing question. The evaluator should confirm that chunks are sufficient for the passing question and flag a problem for the failing one.

In [ ]:
passes = [r for r in results if r["classification"] == "pass"]

# Test on a passing question
pass_q = passes[0]
pass_retrieval = collection.query(
    query_texts=[pass_q["question"]], n_results=3, include=["documents"]
)
pass_eval = evaluate_retrieval(
    pass_q["question"], pass_retrieval["documents"][0], client, model_id
)

print("PASSING QUESTION")
print(f"  Q: {pass_q['question']}")
print(f"  Evaluator verdict: {json.dumps(pass_eval, indent=2)}")

# Test on a failing question
fail_q = failures[0]
fail_retrieval = collection.query(
    query_texts=[fail_q["question"]], n_results=3, include=["documents"]
)
fail_eval = evaluate_retrieval(
    fail_q["question"], fail_retrieval["documents"][0], client, model_id
)

print("\nFAILING QUESTION")
print(f"  Q: {fail_q['question']}")
print(f"  Evaluator verdict: {json.dumps(fail_eval, indent=2)}")

The same model that produced the wrong answer in the passive pipeline can *detect* that the retrieval was insufficient. It always could. The passive architecture simply never asked.

This is the core insight behind agentic RAG: the model is not the bottleneck. The control structure is.

## 2.3 Query Rewriting

When the evaluator flags a retrieval as `partial` or `irrelevant`, the agent needs a recovery action. The simplest one: rewrite the query and try again.

The original question is written in natural language by a user. The vector store responds best to queries that match the terminology and phrasing used in the source documents. A rewritten query bridges that gap.

This is not prompt engineering. It is the agent choosing to reformulate its search before committing to an answer.

In [ ]:
REWRITER_PROMPT = """You are a search query rewriter for a tabletop RPG rules corpus.

The original query did not retrieve useful results. Your job is to rewrite it so
that a vector similarity search is more likely to find the relevant rules text.

Strategies:
- Use terminology that would appear in a rulebook (e.g., "hit die" instead of
  "hit points dice", "ability score modifier" instead of "bonus")
- If the question asks about a specific class, level, or stat, include those
  terms explicitly
- If the question asks about a table or chart, reference the table name
- Break compound questions into the most specific sub-question

You will also receive the REASON the previous retrieval was judged insufficient.
Use that information to guide your rewrite.

Respond with EXACTLY one JSON object (no other text):
{"rewritten_query": "<your rewritten query>", "strategy": "<one sentence explaining what you changed>"}
"""


def rewrite_query(question, eval_result, model_client, model_id):
    """Rewrite a query based on the evaluator's feedback."""
    reason = eval_result.get("reason", "Retrieval was insufficient.")
    missing = eval_result.get("missing", "")

    context = f"ORIGINAL QUERY:\n{question}\n\nREASON RETRIEVAL FAILED:\n{reason}"
    if missing:
        context += f"\nMISSING INFORMATION:\n{missing}"

    response = model_client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": REWRITER_PROMPT},
            {"role": "user", "content": context}
        ],
        temperature=0.0
    )

    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"rewritten_query": question, "strategy": "parse_error", "raw": raw}


print("Query rewriter defined.")

In [ ]:
# Demonstrate query rewriting on a failing question
fail_q = failures[1]  # saving throw question
print(f"Original query : {fail_q['question']}")

# First, retrieve and evaluate
retrieval_1 = collection.query(
    query_texts=[fail_q["question"]], n_results=3, include=["documents"]
)
eval_1 = evaluate_retrieval(
    fail_q["question"], retrieval_1["documents"][0], client, model_id
)
print(f"Eval verdict   : {eval_1['verdict']}")
print(f"Eval reason    : {eval_1.get('reason', '')}")

# Rewrite
rewrite = rewrite_query(fail_q["question"], eval_1, client, model_id)
print(f"\nRewritten query: {rewrite['rewritten_query']}")
print(f"Strategy       : {rewrite['strategy']}")

# Retrieve again with the rewritten query
retrieval_2 = collection.query(
    query_texts=[rewrite["rewritten_query"]], n_results=3, include=["documents"]
)
eval_2 = evaluate_retrieval(
    fail_q["question"], retrieval_2["documents"][0], client, model_id
)
print(f"\nRe-eval verdict: {eval_2['verdict']}")
print(f"Re-eval reason : {eval_2.get('reason', '')}")

Notice what just happened. The model identified what was missing, rewrote the query to target it, and the second retrieval pulled different chunks. The model did not get smarter. The retriever did not get better. The system asked a better question.

This is the first sign that the architecture, not the components, was the bottleneck.

## 2.4 The Agent Loop

Now we have two capabilities the passive pipeline lacked:

1. **Evaluate** whether retrieval was sufficient
2. **Rewrite** the query and try again when it was not

The agent loop connects these into a decision cycle:

```
   Question arrives
        |
        v
   RETRIEVE chunks
        |
        v
   EVALUATE: sufficient?
       / \
     yes   no
      |     |
      v     v
   ANSWER  REWRITE query
      |     |
      v     v
   done   RETRIEVE again
            |
            v
         EVALUATE: sufficient?
            / \
          yes   no (max retries)
           |     |
           v     v
        ANSWER  ABSTAIN
```

The loop has a maximum number of iterations. If the agent cannot find sufficient context after all retries, it says so. That is better than a confident wrong answer.

The cell below implements this loop. Read the code before running it.

In [ ]:
ANSWER_PROMPT = """You are a rules assistant for Basic Fantasy RPG.
Use only the context below to answer the question.
If the context does not contain enough information, say so clearly.

Context:
{context}

Question: {question}"""


def agent_rag(question, collection, model_client, model_id,
              n_results=3, max_retries=2, verbose=True):
    """
    The agent loop: retrieve, evaluate, decide.

    Unlike passive_rag, this function can:
    - Judge whether retrieval was sufficient before answering
    - Rewrite the query and retry when retrieval is weak
    - Abstain when it cannot find sufficient context
    """
    trace = []  # Record every step for inspection
    current_query = question

    for attempt in range(1, max_retries + 2):  # +2 because range is exclusive and we start at 1
        # STEP 1: Retrieve
        retrieved = collection.query(
            query_texts=[current_query],
            n_results=n_results,
            include=["documents", "distances"]
        )
        chunks = retrieved["documents"][0]
        distances = retrieved["distances"][0]

        step = {
            "attempt": attempt,
            "query": current_query,
            "distances": distances
        }

        if verbose:
            print(f"\n--- Attempt {attempt} ---")
            print(f"Query: {current_query}")
            print(f"Distances: {[f'{d:.4f}' for d in distances]}")

        # STEP 2: Evaluate
        eval_result = evaluate_retrieval(question, chunks, model_client, model_id)
        step["eval"] = eval_result

        if verbose:
            print(f"Verdict: {eval_result['verdict']}")
            print(f"Reason: {eval_result.get('reason', '')}")

        # STEP 3: Decide
        if eval_result["verdict"] == "sufficient":
            # Answer
            context = "\n\n".join(chunks)
            prompt = ANSWER_PROMPT.format(context=context, question=question)
            response = model_client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            answer = response.choices[0].message.content
            step["action"] = "answer"
            step["answer"] = answer
            trace.append(step)

            if verbose:
                print(f"Action: ANSWER")

            return {
                "answer": answer,
                "status": "answered",
                "attempts": attempt,
                "trace": trace
            }

        elif attempt <= max_retries:
            # Rewrite and retry
            rewrite = rewrite_query(question, eval_result, model_client, model_id)
            current_query = rewrite["rewritten_query"]
            step["action"] = "rewrite"
            step["rewrite"] = rewrite
            trace.append(step)

            if verbose:
                print(f"Action: REWRITE -> {current_query}")

        else:
            # Exhausted retries, answer with best available context
            context = "\n\n".join(chunks)
            prompt = ANSWER_PROMPT.format(context=context, question=question)
            response = model_client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            answer = response.choices[0].message.content
            step["action"] = "answer_with_caveat"
            step["answer"] = answer
            trace.append(step)

            if verbose:
                print(f"Action: ANSWER (best effort after {attempt} attempts)")

            return {
                "answer": answer,
                "status": "best_effort",
                "attempts": attempt,
                "trace": trace
            }


print("Agent loop defined.")

## 2.5 Testing the Agent Loop

Run the agent loop on each of the failing questions. Compare the agent's answers to both the passive pipeline's output and the expected answers.

Watch the trace. The important thing is not just whether the final answer is better. It is *how the agent got there*: how many retrieval attempts it made, what queries it rewrote, and what decisions it made along the way.

In [ ]:
agent_results = []

for f_ in failures:
    print("=" * 70)
    print(f"QUESTION: {f_['question']}")
    print(f"EXPECTED: {f_['expected']}")
    print(f"PASSIVE PIPELINE GOT: {f_['answer'][:150]}...")
    print()

    result = agent_rag(
        question=f_["question"],
        collection=collection,
        model_client=client,
        model_id=model_id,
        max_retries=2,
        verbose=True
    )

    print(f"\nAGENT ANSWER: {result['answer'][:300]}")
    print(f"STATUS: {result['status']} (after {result['attempts']} attempt(s))")
    print()

    agent_results.append({
        "id": f_["id"],
        "question": f_["question"],
        "expected": f_["expected"],
        "passive_answer": f_["answer"],
        "agent_answer": result["answer"],
        "status": result["status"],
        "attempts": result["attempts"],
        "trace": result["trace"]
    })

## 2.6 Scoring the Agent

Use the same model to classify whether the agent's answers are correct. This is the same evaluation method from the Escalation Lab, applied to the agent's output.

In [ ]:
JUDGE_PROMPT = """You are an evaluation judge. Compare the EXPECTED answer to the ACTUAL answer.

The ACTUAL answer is correct if it conveys the same key facts as the EXPECTED answer,
even if the wording differs. Minor omissions of non-essential details are acceptable.

Respond with EXACTLY one JSON object:
{"classification": "pass" or "fail", "reason": "<one sentence>"}
"""


def judge_answer(question, expected, actual, model_client, model_id):
    """Judge whether the agent's answer matches the expected answer."""
    response = model_client.chat.completions.create(
        model=model_id,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"QUESTION: {question}\n\nEXPECTED: {expected}\n\nACTUAL: {actual}"}
        ],
        temperature=0.0
    )
    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"classification": "error", "reason": "Could not parse judge response", "raw": raw}


print("Scoring agent results...\n")
print(f"{'ID':<6} {'Passive':<10} {'Agent':<10} {'Attempts':<10} {'Question'}")
print("-" * 80)

agent_passes = 0
for ar in agent_results:
    judgment = judge_answer(
        ar["question"], ar["expected"], ar["agent_answer"], client, model_id
    )
    ar["agent_classification"] = judgment["classification"]
    ar["judge_reason"] = judgment.get("reason", "")

    if judgment["classification"] == "pass":
        agent_passes += 1

    print(f"{ar['id']:<6} {'fail':<10} {judgment['classification']:<10} {ar['attempts']:<10} {ar['question'][:50]}")

print(f"\nAgent recovered {agent_passes} of {len(failures)} previously failing questions.")

## 2.7 Comparing Architectures

The table below shows the full picture: all 10 questions, passive vs. agent.

In [ ]:
# Build the combined results
agent_lookup = {ar["id"]: ar for ar in agent_results}

print(f"{'ID':<6} {'Passive':<10} {'Agent':<10} {'Question'}")
print("=" * 80)

passive_total = sum(1 for r in results if r["classification"] == "pass")
agent_total = passive_total  # Start with the questions that already passed

for r in results:
    passive_result = r["classification"]
    if r["id"] in agent_lookup:
        agent_result = agent_lookup[r["id"]]["agent_classification"]
        if agent_result == "pass":
            agent_total += 1
    else:
        agent_result = "pass"  # Already passed in passive pipeline

    passive_display = "pass" if passive_result == "pass" else "FAIL"
    agent_display = "pass" if agent_result == "pass" else "FAIL"

    print(f"{r['id']:<6} {passive_display:<10} {agent_display:<10} {r['question'][:50]}")

print("=" * 80)
print(f"{'TOTAL':<6} {passive_total}/10{'':<5} {agent_total}/10")

The model did not change. The corpus did not change. The retriever did not change. The only difference is the control structure around them.

This is the argument for agentic RAG: the components were already capable. The passive architecture just never gave them the opportunity to recover from failure.

## 2.8 Inspecting the Agent Trace

The trace recorded every decision the agent made. This is critical for production systems: you need to know *why* the agent answered the way it did, not just *what* it answered.

Pick a question and walk through its trace.

In [ ]:
# Inspect the trace for the first agent result
inspect = agent_results[0]

print(f"Question: {inspect['question']}")
print(f"Expected: {inspect['expected']}")
print(f"Status  : {inspect['status']}")
print()

for step in inspect["trace"]:
    print(f"--- Attempt {step['attempt']} ---")
    print(f"  Query    : {step['query']}")
    print(f"  Distances: {[f'{d:.4f}' for d in step['distances']]}")
    print(f"  Verdict  : {step['eval']['verdict']}")
    print(f"  Reason   : {step['eval'].get('reason', '')}")
    print(f"  Action   : {step['action']}")
    if step['action'] == 'rewrite':
        print(f"  Rewrite  : {step['rewrite']['rewritten_query']}")
        print(f"  Strategy : {step['rewrite']['strategy']}")
    elif 'answer' in step:
        print(f"  Answer   : {step['answer'][:200]}")
    print()

## Pre-Built Output

If the cells above did not execute due to endpoint availability or time constraints, run the cell below to load pre-built results and continue the discussion.

In [ ]:
USE_PREBUILT = False  # Set to True if live execution was not available

if USE_PREBUILT:
    prebuilt_path = "../prebuilt/section2_outputs.json"
    try:
        with open(prebuilt_path, "r", encoding="utf-8") as f:
            prebuilt = json.load(f)
        agent_results = prebuilt["agent_results"]
        print("Loaded pre-built Section 2 outputs")
        print(f"Agent results: {len(agent_results)}")
        for ar in agent_results:
            print(f"  {ar['id']}: {ar.get('agent_classification', 'not scored')} "
                  f"({ar['attempts']} attempt(s))")
    except FileNotFoundError:
        print(f"Pre-built file not found at {prebuilt_path}")
        print("Run the cells above or check the prebuilt directory.")
else:
    print("Using live results.")

---

## Key Takeaways

1. **The passive pipeline's failures were architectural, not model failures.** The same model that produced wrong answers could detect bad retrieval and recover from it when given the opportunity.

2. **The agent loop adds three capabilities:** evaluate retrieval quality, rewrite queries, and decide when to abstain. None of these require a different model or different data.

3. **The trace is the audit trail.** Every decision is recorded. In production, this is how you debug, monitor, and improve the system.

4. **Complexity was added only where evidence justified it.** The passive pipeline worked for 6 of 10 questions. The agent loop was built specifically to address the 4 it could not handle.

---

> **FIELD TAKEAWAY**
>
> An agent loop is not a better model. It is a better control structure. The model evaluates its own retrieval, rewrites its own queries, and decides when to answer or abstain. The components stay the same. The architecture changes. That is the difference between passive and agentic RAG.